Environment setup
Mount Drive
Load train/dev/test data
Train/load Arabic and English SentencePiece tokenizers
Build scratch Transformer
Train AR→EN
Evaluate AR→EN
Train EN→AR
Evaluate EN→AR
Save BLEU / chrF / SBERT and predictions

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# import os

# print("Exists:", os.path.exists("/content/drive"))
# print("Contents:", os.listdir("/content/drive")[:20])

In [ ]:
# from pathlib import Path

# base = Path("/content/drive/MyDrive/Remaining_Evaluating_Transformer")

# print("Base exists:", base.exists())
# print("DataSet exists:", (base / "DataSet").exists())

In [ ]:
# import shutil
# from pathlib import Path

# drive_path = Path("/content/drive")

# for item in drive_path.iterdir():
#     if item.is_dir():
#         shutil.rmtree(item)
#     else:
#         item.unlink()

# print("Cleaned:", list(drive_path.iterdir()))

In [ ]:
!pip -q install sentencepiece

In [ ]:
import sentencepiece as spm
from pathlib import Path

SP_DIR = Path("/content/drive/MyDrive/Remaining_Evaluating_Transformer/Tokenizers/scratch_sentencepiece")
SP_DIR.mkdir(parents=True, exist_ok=True)

AR_TRAIN_TXT = "/content/drive/MyDrive/Remaining_Evaluating_Transformer/DataSet/opus.ar-en-train.ar"
EN_TRAIN_TXT = "/content/drive/MyDrive/Remaining_Evaluating_Transformer/DataSet/opus.ar-en-train.en"

In [ ]:
# spm.SentencePieceTrainer.train(
#     input=AR_TRAIN_TXT,
#     model_prefix=str(SP_DIR / "arabic_sp32k"),
#     vocab_size=32000,
#     model_type="bpe",
#     character_coverage=1.0,
#     bos_id=2,
#     eos_id=3,
#     pad_id=0,
#     unk_id=1,
# )

KeyboardInterrupt: 

In [ ]:
# spm.SentencePieceTrainer.train(
#     input=EN_TRAIN_TXT,
#     model_prefix=str(SP_DIR / "english_sp32k"),
#     vocab_size=32000,
#     model_type="bpe",
#     character_coverage=1.0,
#     bos_id=2,
#     eos_id=3,
#     pad_id=0,
#     unk_id=1,
# )

True

In [ ]:
import sentencepiece as spm

sp_ar = spm.SentencePieceProcessor(
    model_file=str(SP_DIR / "arabic_sp32k.model")
)

sp_en = spm.SentencePieceProcessor(
    model_file=str(SP_DIR / "english_sp32k.model")
)

print("Arabic vocab:", sp_ar.vocab_size())
print("English vocab:", sp_en.vocab_size())

print(sp_ar.encode("هذا اختبار بسيط للترجمة", out_type=str))
print(sp_en.encode("This is a simple translation test.", out_type=str))

Arabic vocab: 32000
English vocab: 32000
['▁هذا', '▁اختبار', '▁بسيط', '▁للت', 'رجمة']
['▁This', '▁is', '▁a', '▁simple', '▁translation', '▁test', '.']


In [ ]:
print("Arabic:")
print("PAD:", sp_ar.pad_id())
print("UNK:", sp_ar.unk_id())
print("BOS:", sp_ar.bos_id())
print("EOS:", sp_ar.eos_id())

print("\nEnglish:")
print("PAD:", sp_en.pad_id())
print("UNK:", sp_en.unk_id())
print("BOS:", sp_en.bos_id())
print("EOS:", sp_en.eos_id())

Arabic:
PAD: 0
UNK: 1
BOS: 2
EOS: 3

English:
PAD: 0
UNK: 1
BOS: 2
EOS: 3


In [ ]:
MAX_LEN = 256

def encode_ar(text):
    ids = sp_ar.encode(text, out_type=int)

    ids = (
        [sp_ar.bos_id()]
        + ids[:MAX_LEN - 2]
        + [sp_ar.eos_id()]
    )

    return ids


def encode_en(text):
    ids = sp_en.encode(text, out_type=int)

    ids = (
        [sp_en.bos_id()]
        + ids[:MAX_LEN - 2]
        + [sp_en.eos_id()]
    )

    return ids

In [ ]:
ar_example = "هذا اختبار بسيط للترجمة"
en_example = "This is a simple translation test."

ar_ids = encode_ar(ar_example)
en_ids = encode_en(en_example)

print("AR IDs:", ar_ids)
print("AR decoded:", sp_ar.decode(ar_ids))

print()

print("EN IDs:", en_ids)
print("EN decoded:", sp_en.decode(en_ids))

AR IDs: [2, 72, 8562, 7489, 447, 4649, 3]
AR decoded: هذا اختبار بسيط للترجمة

EN IDs: [2, 425, 86, 6, 4506, 11238, 2108, 31437, 3]
EN decoded: This is a simple translation test.


In [ ]:
import torch
from torch.utils.data import Dataset

class TranslationDataset(Dataset):
    def __init__(self, src_lines, tgt_lines, src_encoder, tgt_encoder):
        assert len(src_lines) == len(tgt_lines)

        self.src_lines = src_lines
        self.tgt_lines = tgt_lines
        self.src_encoder = src_encoder
        self.tgt_encoder = tgt_encoder

    def __len__(self):
        return len(self.src_lines)

    def __getitem__(self, idx):
        return {
            "src": torch.tensor(
                self.src_encoder(self.src_lines[idx]),
                dtype=torch.long
            ),
            "tgt": torch.tensor(
                self.tgt_encoder(self.tgt_lines[idx]),
                dtype=torch.long
            ),
        }

In [ ]:
from torch.nn.utils.rnn import pad_sequence

PAD_ID = 0

def collate_fn(batch):
    src = [x["src"] for x in batch]
    tgt = [x["tgt"] for x in batch]

    src = pad_sequence(
        src,
        batch_first=True,
        padding_value=PAD_ID
    )

    tgt = pad_sequence(
        tgt,
        batch_first=True,
        padding_value=PAD_ID
    )

    return {
        "src": src,
        "tgt": tgt,
    }

In [ ]:
from pathlib import Path

DATA_DIR = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/DataSet"
)

def read_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]

train_ar = read_lines(DATA_DIR / "opus.ar-en-train.ar")
train_en = read_lines(DATA_DIR / "opus.ar-en-train.en")

dev_ar = read_lines(DATA_DIR / "opus.ar-en-dev.ar")
dev_en = read_lines(DATA_DIR / "opus.ar-en-dev.en")

test_ar = read_lines(DATA_DIR / "opus.ar-en-test.ar")
test_en = read_lines(DATA_DIR / "opus.ar-en-test.en")

print(len(train_ar), len(train_en))
print(len(dev_ar), len(dev_en))
print(len(test_ar), len(test_en))

1000000 1000000
2000 2000
2000 2000


In [ ]:
train_dataset = TranslationDataset(
    train_ar,
    train_en,
    encode_ar,
    encode_en,
)

dev_dataset = TranslationDataset(
    dev_ar,
    dev_en,
    encode_ar,
    encode_en,
)

test_dataset = TranslationDataset(
    test_ar,
    test_en,
    encode_ar,
    encode_en,
)

In [ ]:
sample = train_dataset[0]

print("SRC shape:", sample["src"].shape)
print("TGT shape:", sample["tgt"].shape)

print("SRC:", sp_ar.decode(sample["src"].tolist()))
print("TGT:", sp_en.decode(sample["tgt"].tolist()))

SRC shape: torch.Size([5])
TGT shape: torch.Size([5])
SRC: و هذه؟
TGT: And this?


In [ ]:
import math
import torch
import torch.nn as nn


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(
            0, max_len, dtype=torch.float
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0, d_model, 2, dtype=torch.float
            )
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class ScratchTransformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=512,
        nhead=8,
        num_encoder_layers=6,
        num_decoder_layers=6,
        dim_feedforward=2048,
        dropout=0.1,
        pad_id=0,
    ):
        super().__init__()

        self.d_model = d_model
        self.pad_id = pad_id

        self.src_embedding = nn.Embedding(
            src_vocab_size,
            d_model,
            padding_idx=pad_id
        )

        self.tgt_embedding = nn.Embedding(
            tgt_vocab_size,
            d_model,
            padding_idx=pad_id
        )

        self.positional_encoding = PositionalEncoding(
            d_model=d_model,
            max_len=512
        )

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )

        self.output_projection = nn.Linear(
            d_model,
            tgt_vocab_size
        )

    def generate_square_subsequent_mask(
        self,
        size,
        device
    ):
        # Boolean causal mask.
        # True = position is masked.
        return torch.triu(
            torch.ones(
                size,
                size,
                dtype=torch.bool,
                device=device
            ),
            diagonal=1
        )

    def forward(self, src, tgt_input):

        src_pad_mask = src.eq(self.pad_id)
        tgt_pad_mask = tgt_input.eq(self.pad_id)

        tgt_mask = self.generate_square_subsequent_mask(
            tgt_input.size(1),
            tgt_input.device
        )

        src_emb = self.src_embedding(src)
        src_emb = src_emb * math.sqrt(self.d_model)
        src_emb = self.positional_encoding(src_emb)

        tgt_emb = self.tgt_embedding(tgt_input)
        tgt_emb = tgt_emb * math.sqrt(self.d_model)
        tgt_emb = self.positional_encoding(tgt_emb)

        output = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_pad_mask,
            tgt_key_padding_mask=tgt_pad_mask,
            memory_key_padding_mask=src_pad_mask,
        )

        logits = self.output_projection(output)

        return logits

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model = ScratchTransformer(
    src_vocab_size=sp_ar.vocab_size(),
    tgt_vocab_size=sp_en.vocab_size(),
    d_model=512,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=2048,
    dropout=0.1,
    pad_id=0,
).to(device)

print(
    "Parameters:",
    f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M"
)

Device: cuda
GPU: Tesla T4


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


Parameters: 93.32M


In [ ]:
from torch.utils.data import DataLoader

TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
)

In [ ]:
criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.98),
    eps=1e-9,
    weight_decay=0.01,
)

In [ ]:
batch = next(iter(train_loader))

src = batch["src"].to(device)
tgt = batch["tgt"].to(device)

tgt_input = tgt[:, :-1]
tgt_output = tgt[:, 1:]

with torch.no_grad():
    logits = model(
        src,
        tgt_input
    )

print("SRC:", src.shape)
print("TGT input:", tgt_input.shape)
print("Logits:", logits.shape)
print(
    "Expected last dimension:",
    sp_en.vocab_size()
)

SRC: torch.Size([32, 82])
TGT input: torch.Size([32, 75])
Logits: torch.Size([32, 75, 32000])
Expected last dimension: 32000


In [ ]:
import os
import json
import time
import math
import torch
from pathlib import Path
from tqdm.auto import tqdm

EPOCHS = 1
PAD_ID = 0
CHECKPOINT_EVERY = 2500

LOCAL_CKPT_DIR = Path(
    "/content/scratch_sp_ar_en/checkpoints"
)

DRIVE_RESULT_DIR = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Results/"
    "scratch_sp_ar_en"
)

LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULT_DIR.mkdir(parents=True, exist_ok=True)

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.98),
    eps=1e-9,
    weight_decay=0.01,
)

In [ ]:
def evaluate_loss(model, loader):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for batch in tqdm(
            loader,
            desc="Validation",
            leave=False
        ):
            src = batch["src"].to(device)
            tgt = batch["tgt"].to(device)

            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]

            logits = model(
                src,
                tgt_input
            )

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                tgt_output.reshape(-1)
            )

            nonpad = tgt_output.ne(PAD_ID).sum().item()

            total_loss += loss.item() * nonpad
            total_tokens += nonpad

    model.train()

    return total_loss / total_tokens

In [ ]:
def save_checkpoint(
    model,
    optimizer,
    step,
    epoch,
    train_loss,
    dev_loss
):
    ckpt = {
        "step": step,
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "dev_loss": dev_loss,
    }

    local_path = (
        LOCAL_CKPT_DIR /
        f"checkpoint_{step}.pt"
    )

    drive_path = (
        DRIVE_RESULT_DIR /
        f"checkpoint_{step}.pt"
    )

    torch.save(
        ckpt,
        local_path
    )

    # Copy only after local save succeeds
    import shutil
    shutil.copy2(
        local_path,
        drive_path
    )

    print(
        f"\nCheckpoint {step} saved:"
        f"\n  local: {local_path}"
        f"\n  drive: {drive_path}"
    )

In [ ]:
global_step = 0
running_loss = 0.0
running_tokens = 0

history = []

model.train()

for epoch in range(EPOCHS):

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for batch in progress:

        src = batch["src"].to(device)
        tgt = batch["tgt"].to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        optimizer.zero_grad()

        logits = model(
            src,
            tgt_input
        )

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_output.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        global_step += 1

        nonpad = tgt_output.ne(PAD_ID).sum().item()

        running_loss += loss.item() * nonpad
        running_tokens += nonpad

        mean_train_loss = (
            running_loss / running_tokens
        )

        progress.set_postfix({
            "step": global_step,
            "loss": f"{mean_train_loss:.4f}"
        })

        if global_step % CHECKPOINT_EVERY == 0:

            dev_loss = evaluate_loss(
                model,
                dev_loader
            )

            print(
                f"\nStep {global_step}"
                f"\nTrain token loss: "
                f"{mean_train_loss:.6f}"
                f"\nDev token loss: "
                f"{dev_loss:.6f}"
                f"\nDev perplexity: "
                f"{math.exp(dev_loss):.2f}"
            )

            history.append({
                "step": global_step,
                "train_loss": mean_train_loss,
                "dev_loss": dev_loss,
                "dev_perplexity": math.exp(dev_loss),
            })

            save_checkpoint(
                model=model,
                optimizer=optimizer,
                step=global_step,
                epoch=epoch + 1,
                train_loss=mean_train_loss,
                dev_loss=dev_loss
            )

Epoch 1/1:   0%|          | 0/31250 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import torch
from pathlib import Path

print("CUDA available:", torch.cuda.is_available())
print("Device:", device if "device" in globals() else "device variable missing")
print("Global step:", global_step if "global_step" in globals() else "global_step missing")

print("\nLocal checkpoints:")
for p in sorted(Path("/content/scratch_sp_ar_en/checkpoints").glob("checkpoint_*.pt")):
    print(p.name, f"{p.stat().st_size / 1024**2:.1f} MB")

print("\nDrive checkpoints:")
drive_dir = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Results/"
    "scratch_sp_ar_en"
)

if drive_dir.exists():
    for p in sorted(drive_dir.glob("checkpoint_*.pt")):
        print(p.name, f"{p.stat().st_size / 1024**2:.1f} MB")

CUDA available: True
Device: cuda
Global step: 19

Local checkpoints:

Drive checkpoints:
checkpoint_10000.pt 1069.2 MB
checkpoint_12500.pt 1069.2 MB
checkpoint_15000.pt 1069.2 MB
checkpoint_17500.pt 1069.2 MB
checkpoint_20000.pt 1069.2 MB
checkpoint_22500.pt 1069.2 MB
checkpoint_2500.pt 1069.2 MB
checkpoint_25000.pt 1069.2 MB
checkpoint_27500.pt 1069.2 MB
checkpoint_30000.pt 1069.2 MB
checkpoint_31250.pt 1069.2 MB
checkpoint_5000.pt 1069.2 MB
checkpoint_7500.pt 1069.2 MB


In [ ]:
import torch
import torch.nn as nn
from pathlib import Path

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [ ]:
model = ScratchTransformer(
    src_vocab_size=sp_ar.vocab_size(),
    tgt_vocab_size=sp_en.vocab_size(),
    d_model=512,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=2048,
    dropout=0.1,
    pad_id=0,
).to(device)

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.98),
    eps=1e-9,
    weight_decay=0.01,
)

In [ ]:
CKPT_PATH = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Results/"
    "scratch_sp_ar_en/checkpoint_30000.pt"
)

checkpoint = torch.load(
    CKPT_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

global_step = checkpoint["step"]

print("Restored step:", global_step)
print("Epoch:", checkpoint["epoch"])
print("Train loss:", checkpoint["train_loss"])
print("Dev loss:", checkpoint["dev_loss"])

Restored step: 30000
Epoch: 1
Train loss: 4.278184071787503
Dev loss: 3.8524174517358074


In [ ]:
TARGET_STEPS = 31250

model.train()

running_loss = 0.0
running_tokens = 0

progress = tqdm(
    train_loader,
    desc=f"Resume {global_step} → {TARGET_STEPS}"
)

for batch in progress:

    if global_step >= TARGET_STEPS:
        break

    src = batch["src"].to(device)
    tgt = batch["tgt"].to(device)

    tgt_input = tgt[:, :-1]
    tgt_output = tgt[:, 1:]

    optimizer.zero_grad()

    logits = model(
        src,
        tgt_input
    )

    loss = criterion(
        logits.reshape(-1, logits.size(-1)),
        tgt_output.reshape(-1)
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        1.0
    )

    optimizer.step()

    global_step += 1

    nonpad = tgt_output.ne(PAD_ID).sum().item()

    running_loss += loss.item() * nonpad
    running_tokens += nonpad

    mean_train_loss = (
        running_loss / running_tokens
    )

    progress.set_postfix({
        "step": global_step,
        "loss": f"{mean_train_loss:.4f}"
    })

Resume 30000 → 31250:   0%|          | 0/31250 [00:00<?, ?it/s]

In [ ]:
dev_loss = evaluate_loss(
    model,
    dev_loader
)

print("FINAL STEP:", global_step)
print("Final train token loss:", mean_train_loss)
print("Final dev token loss:", dev_loss)
print("Final dev perplexity:", math.exp(dev_loss))

Validation:   0%|          | 0/63 [00:00<?, ?it/s]

FINAL STEP: 30000
Final train token loss: 9.22021745613871
Final dev token loss: 3.8524174517358074
Final dev perplexity: 47.10680412002097


In [ ]:
save_checkpoint(
    model=model,
    optimizer=optimizer,
    step=global_step,
    epoch=1,
    train_loss=mean_train_loss,
    dev_loss=dev_loss
)


Checkpoint 30000 saved:
  local: /content/scratch_sp_ar_en/checkpoints/checkpoint_30000.pt
  drive: /content/drive/MyDrive/Remaining_Evaluating_Transformer/Results/scratch_sp_ar_en/checkpoint_30000.pt


In [ ]:
from pathlib import Path
import torch

FINAL_CKPT = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Results/"
    "scratch_sp_ar_en/checkpoint_31250.pt"
)

checkpoint = torch.load(
    FINAL_CKPT,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Loaded step:", checkpoint["step"])

Loaded step: 31250


In [ ]:
import torch
from tqdm.auto import tqdm

BOS_ID = sp_en.bos_id()
EOS_ID = sp_en.eos_id()
PAD_ID = sp_en.pad_id()

MAX_GEN_LEN = 256
BEAM_SIZE = 4

In [ ]:
@torch.no_grad()
def beam_search_translate(
    model,
    src,
    beam_size=4,
    max_len=256
):
    model.eval()

    # Start with BOS
    beams = [
        (
            torch.tensor(
                [[BOS_ID]],
                dtype=torch.long,
                device=device
            ),
            0.0
        )
    ]

    completed = []

    for _ in range(max_len - 1):

        candidates = []

        for seq, score in beams:

            if seq[0, -1].item() == EOS_ID:
                completed.append((seq, score))
                continue

            logits = model(
                src,
                seq
            )

            log_probs = torch.log_softmax(
                logits[:, -1, :],
                dim=-1
            )

            top_scores, top_ids = torch.topk(
                log_probs,
                beam_size,
                dim=-1
            )

            for k in range(beam_size):

                token_id = top_ids[0, k].view(1, 1)

                new_seq = torch.cat(
                    [seq, token_id],
                    dim=1
                )

                new_score = (
                    score
                    + top_scores[0, k].item()
                )

                candidates.append(
                    (new_seq, new_score)
                )

        if not candidates:
            break

        # Length-normalized ranking
        candidates.sort(
            key=lambda x:
                x[1] / (x[0].size(1) ** 0.7),
            reverse=True
        )

        beams = candidates[:beam_size]

        if len(completed) >= beam_size:
            break

    completed.extend(beams)

    completed.sort(
        key=lambda x:
        x[1] / (x[0].size(1) ** 0.7),
        reverse=True
    )

    best = completed[0][0][0].tolist()

    # Remove BOS/EOS
    if best and best[0] == BOS_ID:
        best = best[1:]

    if EOS_ID in best:
        best = best[:best.index(EOS_ID)]

    return sp_en.decode(best)

In [ ]:
predictions = []

model.eval()

for i in tqdm(
    range(len(test_ar)),
    desc="AR→EN SentencePiece evaluation"
):
    src_ids = encode_ar(test_ar[i])

    src = torch.tensor(
        [src_ids],
        dtype=torch.long,
        device=device
    )

    prediction = beam_search_translate(
        model,
        src,
        beam_size=4,
        max_len=256
    )

    predictions.append(prediction)

print("Predictions:", len(predictions))

AR→EN SentencePiece evaluation:   0%|          | 0/2000 [00:00<?, ?it/s]

Predictions: 2000


In [ ]:
for i in range(10):
    print("=" * 90)
    print("SRC:", test_ar[i])
    print("REF:", test_en[i])
    print("HYP:", predictions[i])

SRC: حسناً ، تبعاً للتقرير هناك ثلاث عبوات ماء مفقودة
REF: Well, according to the report, there were three water bottles missing.
HYP: All right, there's three left.
SRC: 81 - استعرضت اللجنة البيانات التي طلبتها من المؤسسات عن صعوبات التوظيف والاستبقاء.
REF: The Commission reviewed the data on recruitment and retention difficulties, which it had requested from the organizations.
HYP: The Committee took note of the data on the basis of the institutions that have been used.
SRC: هل سترجع للعيش معنا هنا
REF: (Maya) So Are You Gonna Move Back In,
HYP: Will you stay here with us?
SRC: أجل، تستطيع
REF: Yes, you may.
HYP: Yeah, you can.
SRC: مما أنت خائف؟
REF: What are you scared of?
HYP: So you're afraid?
SRC: {\cH1F20B1\3cHFFFFFF}: كما يقول زوجي دائماً
REF: Like my husband always says...
HYP: My husband says:
SRC: هيّا يا مهدّئات العضلات.
REF: Come on, muscle relaxers.
HYP: Come on, come on.
SRC: United Nations Juridical Yearbook, 1965, p.
REF: United Nations Juridical Yearbook, 1965, p. 41

In [ ]:
pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [sacrebleu]


Evaluation

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import math
import torch
import torch.nn as nn
import sentencepiece as spm
from pathlib import Path
from tqdm.auto import tqdm

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
SP_DIR = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Tokenizers/"
    "scratch_sentencepiece"
)

sp_ar = spm.SentencePieceProcessor(
    model_file=str(SP_DIR / "arabic_sp32k.model")
)

sp_en = spm.SentencePieceProcessor(
    model_file=str(SP_DIR / "english_sp32k.model")
)

print(sp_ar.vocab_size(), sp_en.vocab_size())

32000 32000


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(
            0, max_len, dtype=torch.float
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0, d_model, 2, dtype=torch.float
            )
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class ScratchTransformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=512,
        nhead=8,
        num_encoder_layers=6,
        num_decoder_layers=6,
        dim_feedforward=2048,
        dropout=0.1,
        pad_id=0,
    ):
        super().__init__()

        self.d_model = d_model
        self.pad_id = pad_id

        self.src_embedding = nn.Embedding(
            src_vocab_size,
            d_model,
            padding_idx=pad_id
        )

        self.tgt_embedding = nn.Embedding(
            tgt_vocab_size,
            d_model,
            padding_idx=pad_id
        )

        self.positional_encoding = PositionalEncoding(
            d_model
        )

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )

        self.output_projection = nn.Linear(
            d_model,
            tgt_vocab_size
        )

    def generate_square_subsequent_mask(
        self,
        size,
        device
    ):
        return torch.triu(
            torch.ones(
                size,
                size,
                dtype=torch.bool,
                device=device
            ),
            diagonal=1
        )

    def forward(self, src, tgt_input):
        src_pad_mask = src.eq(self.pad_id)
        tgt_pad_mask = tgt_input.eq(self.pad_id)

        tgt_mask = self.generate_square_subsequent_mask(
            tgt_input.size(1),
            tgt_input.device
        )

        src_emb = self.positional_encoding(
            self.src_embedding(src)
            * math.sqrt(self.d_model)
        )

        tgt_emb = self.positional_encoding(
            self.tgt_embedding(tgt_input)
            * math.sqrt(self.d_model)
        )

        output = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_pad_mask,
            tgt_key_padding_mask=tgt_pad_mask,
            memory_key_padding_mask=src_pad_mask,
        )

        return self.output_projection(output)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = ScratchTransformer(
    src_vocab_size=sp_ar.vocab_size(),
    tgt_vocab_size=sp_en.vocab_size(),
    d_model=512,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=2048,
    dropout=0.1,
    pad_id=0,
).to(device)

FINAL_CKPT = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Results/"
    "scratch_sp_ar_en/checkpoint_31250.pt"
)

checkpoint = torch.load(
    FINAL_CKPT,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Loaded step:", checkpoint["step"])
print("Device:", device)

Loaded step: 31250
Device: cuda


In [ ]:
DATA_DIR = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/DataSet"
)

def read_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]

test_ar = read_lines(
    DATA_DIR / "opus.ar-en-test.ar"
)

test_en = read_lines(
    DATA_DIR / "opus.ar-en-test.en"
)

print(len(test_ar), len(test_en))

2000 2000


In [ ]:
MAX_LEN = 256

def encode_ar(text):
    ids = sp_ar.encode(
        text,
        out_type=int
    )

    return (
        [sp_ar.bos_id()]
        + ids[:MAX_LEN - 2]
        + [sp_ar.eos_id()]
    )

In [ ]:
import sacrebleu

bleu = sacrebleu.corpus_bleu(
    predictions,
    [test_en]
)

chrf = sacrebleu.corpus_chrf(
    predictions,
    [test_en]
)

print("FINAL BLEU:", bleu.score)
print("FINAL chrF:", chrf.score)

FINAL BLEU: 8.566542495424986
FINAL chrF: 25.385338575533595


In [ ]:
print("Number of predictions:", len(predictions))
print("Number of references:", len(test_en))

Number of predictions: 2000
Number of references: 2000


In [ ]:
BOS_ID = sp_en.bos_id()
EOS_ID = sp_en.eos_id()

MAX_GEN_LEN = 256
BEAM_SIZE = 4

In [ ]:
@torch.no_grad()
def beam_search_translate(
    model,
    src,
    beam_size=4,
    max_len=256
):
    model.eval()

    beams = [
        (
            torch.tensor(
                [[BOS_ID]],
                dtype=torch.long,
                device=device
            ),
            0.0
        )
    ]

    completed = []

    for _ in range(max_len - 1):

        candidates = []

        for seq, score in beams:

            if seq[0, -1].item() == EOS_ID:
                completed.append((seq, score))
                continue

            logits = model(src, seq)

            log_probs = torch.log_softmax(
                logits[:, -1, :],
                dim=-1
            )

            top_scores, top_ids = torch.topk(
                log_probs,
                beam_size,
                dim=-1
            )

            for k in range(beam_size):

                token_id = top_ids[0, k].view(1, 1)

                new_seq = torch.cat(
                    [seq, token_id],
                    dim=1
                )

                new_score = (
                    score
                    + top_scores[0, k].item()
                )

                candidates.append(
                    (new_seq, new_score)
                )

        if not candidates:
            break

        candidates.sort(
            key=lambda x:
            x[1] / (x[0].size(1) ** 0.7),
            reverse=True
        )

        beams = candidates[:beam_size]

        if len(completed) >= beam_size:
            break

    completed.extend(beams)

    completed.sort(
        key=lambda x:
        x[1] / (x[0].size(1) ** 0.7),
        reverse=True
    )

    best = completed[0][0][0].tolist()

    if best and best[0] == BOS_ID:
        best = best[1:]

    if EOS_ID in best:
        best = best[:best.index(EOS_ID)]

    return sp_en.decode(best)

In [ ]:
src_ids = encode_ar(test_ar[0])

src = torch.tensor(
    [src_ids],
    dtype=torch.long,
    device=device
)

test_prediction = beam_search_translate(
    model,
    src,
    beam_size=4,
    max_len=256
)

print("SRC:", test_ar[0])
print("REF:", test_en[0])
print("HYP:", test_prediction)

SRC: حسناً ، تبعاً للتقرير هناك ثلاث عبوات ماء مفقودة
REF: Well, according to the report, there were three water bottles missing.
HYP: All right, there's three left.


In [ ]:
predictions = []

for i in tqdm(
    range(len(test_ar)),
    desc="AR→EN SentencePiece evaluation"
):
    src_ids = encode_ar(test_ar[i])

    src = torch.tensor(
        [src_ids],
        dtype=torch.long,
        device=device
    )

    prediction = beam_search_translate(
        model,
        src,
        beam_size=4,
        max_len=256
    )

    predictions.append(prediction)

print("Predictions generated:", len(predictions))

AR→EN SentencePiece evaluation:   0%|          | 0/2000 [00:00<?, ?it/s]

Predictions generated: 2000


In [ ]:
import sacrebleu

bleu = sacrebleu.corpus_bleu(
    predictions,
    [test_en]
)

chrf = sacrebleu.corpus_chrf(
    predictions,
    [test_en]
)

print("FINAL BLEU:", bleu.score)
print("FINAL chrF:", chrf.score)

FINAL BLEU: 8.566542495424986
FINAL chrF: 25.385338575533595


In [ ]:
for i in range(20):
    print("=" * 90)
    print("SRC:", test_ar[i])
    print("REF:", test_en[i])
    print("HYP:", predictions[i])

SRC: حسناً ، تبعاً للتقرير هناك ثلاث عبوات ماء مفقودة
REF: Well, according to the report, there were three water bottles missing.
HYP: All right, there's three left.
SRC: 81 - استعرضت اللجنة البيانات التي طلبتها من المؤسسات عن صعوبات التوظيف والاستبقاء.
REF: The Commission reviewed the data on recruitment and retention difficulties, which it had requested from the organizations.
HYP: The Committee took note of the data on the basis of the institutions that have been used.
SRC: هل سترجع للعيش معنا هنا
REF: (Maya) So Are You Gonna Move Back In,
HYP: Will you stay here with us?
SRC: أجل، تستطيع
REF: Yes, you may.
HYP: Yeah, you can.
SRC: مما أنت خائف؟
REF: What are you scared of?
HYP: So you're afraid?
SRC: {\cH1F20B1\3cHFFFFFF}: كما يقول زوجي دائماً
REF: Like my husband always says...
HYP: My husband says:
SRC: هيّا يا مهدّئات العضلات.
REF: Come on, muscle relaxers.
HYP: Come on, come on.
SRC: United Nations Juridical Yearbook, 1965, p.
REF: United Nations Juridical Yearbook, 1965, p. 41

In [ ]:
!pip -q install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    device=str(device)
)

ref_embeddings = sbert_model.encode(
    test_en,
    batch_size=32,
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True,
)

pred_embeddings = sbert_model.encode(
    predictions,
    batch_size=32,
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True,
)

sbert_scores = (
    ref_embeddings * pred_embeddings
).sum(dim=1)

sbert_mean = float(
    sbert_scores.mean().cpu()
)

print("FINAL SBERT:", sbert_mean)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

... (7 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

FINAL SBERT: 0.5286168456077576


In [ ]:
import pandas as pd
from pathlib import Path

RESULT_DIR = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Results/"
    "scratch_sp_ar_en"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

df = pd.DataFrame({
    "source_ar": test_ar,
    "reference_en": test_en,
    "prediction_en": predictions,
})

pred_path = (
    RESULT_DIR /
    "scratch_sp_ar_en_final_predictions.csv"
)

df.to_csv(
    pred_path,
    index=False,
    encoding="utf-8-sig"
)

print("Rows:", len(df))
print("Saved:", pred_path)

Rows: 2000
Saved: /content/drive/MyDrive/Remaining_Evaluating_Transformer/Results/scratch_sp_ar_en/scratch_sp_ar_en_final_predictions.csv


In [ ]:
from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    device=str(device)
)

ref_embeddings = sbert_model.encode(
    test_en,
    batch_size=32,
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True,
)

pred_embeddings = sbert_model.encode(
    predictions,
    batch_size=32,
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True,
)

sbert_scores = (
    ref_embeddings * pred_embeddings
).sum(dim=1)

sbert_mean = float(
    sbert_scores.mean().cpu()
)

print("FINAL SBERT:", sbert_mean)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

FINAL SBERT: 0.5286168456077576


In [ ]:
import json

final_results = {
    "model": "scratch_transformer_sentencepiece_32k",
    "direction": "ar-en",
    "training_steps": 31250,
    "test_size": 2000,
    "bleu": 8.566542495424986,
    "chrf": 25.385338575533595,
    "sbert": 0.5286168456077576,
    "sentencepiece_vocab_src": 32000,
    "sentencepiece_vocab_tgt": 32000,
    "beam_size": 4,
    "max_length": 256,
    "seed": 42,
}

metrics_path = (
    RESULT_DIR /
    "scratch_sp_ar_en_final_metrics.json"
)

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(
        final_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(final_results, indent=2))
print("Saved:", metrics_path)

{
  "model": "scratch_transformer_sentencepiece_32k",
  "direction": "ar-en",
  "training_steps": 31250,
  "test_size": 2000,
  "bleu": 8.566542495424986,
  "chrf": 25.385338575533595,
  "sbert": 0.5286168456077576,
  "sentencepiece_vocab_src": 32000,
  "sentencepiece_vocab_tgt": 32000,
  "beam_size": 4,
  "max_length": 256,
  "seed": 42
}
Saved: /content/drive/MyDrive/Remaining_Evaluating_Transformer/Results/scratch_sp_ar_en/scratch_sp_ar_en_final_metrics.json


In [ ]:
for i in range(15):
    print("=" * 100)
    print("SRC:", test_ar[i])
    print("REF:", test_en[i])
    print("HYP:", predictions[i])

SRC: حسناً ، تبعاً للتقرير هناك ثلاث عبوات ماء مفقودة
REF: Well, according to the report, there were three water bottles missing.
HYP: All right, there's three left.
SRC: 81 - استعرضت اللجنة البيانات التي طلبتها من المؤسسات عن صعوبات التوظيف والاستبقاء.
REF: The Commission reviewed the data on recruitment and retention difficulties, which it had requested from the organizations.
HYP: The Committee took note of the data on the basis of the institutions that have been used.
SRC: هل سترجع للعيش معنا هنا
REF: (Maya) So Are You Gonna Move Back In,
HYP: Will you stay here with us?
SRC: أجل، تستطيع
REF: Yes, you may.
HYP: Yeah, you can.
SRC: مما أنت خائف؟
REF: What are you scared of?
HYP: So you're afraid?
SRC: {\cH1F20B1\3cHFFFFFF}: كما يقول زوجي دائماً
REF: Like my husband always says...
HYP: My husband says:
SRC: هيّا يا مهدّئات العضلات.
REF: Come on, muscle relaxers.
HYP: Come on, come on.
SRC: United Nations Juridical Yearbook, 1965, p.
REF: United Nations Juridical Yearbook, 1965, p. 41

In [ ]:
train_dataset = TranslationDataset(
    train_en,
    train_ar,
    encode_en,
    encode_ar,
)

dev_dataset = TranslationDataset(
    dev_en,
    dev_ar,
    encode_en,
    encode_ar,
)

test_dataset = TranslationDataset(
    test_en,
    test_ar,
    encode_en,
    encode_ar,
)

In [ ]:
model = ScratchTransformer(
    src_vocab_size=sp_en.vocab_size(),
    tgt_vocab_size=sp_ar.vocab_size(),
    d_model=512,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=2048,
    dropout=0.1,
    pad_id=0,
).to(device)

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


In [ ]:
DRIVE_RESULT_DIR = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Results/"
    "scratch_sp_en_ar"
)

In [ ]:
BOS_ID = sp_ar.bos_id()
EOS_ID = sp_ar.eos_id()

MAX_GEN_LEN = 256
BEAM_SIZE = 4

In [ ]:
MAX_LEN = 256

def encode_en(text):
    ids = sp_en.encode(
        text,
        out_type=int
    )

    return (
        [sp_en.bos_id()]
        + ids[:MAX_LEN - 2]
        + [sp_en.eos_id()]
    )

In [ ]:
@torch.no_grad()
def beam_search_translate_en_ar(
    model,
    src,
    beam_size=4,
    max_len=256
):
    model.eval()

    beams = [
        (
            torch.tensor(
                [[BOS_ID]],
                dtype=torch.long,
                device=device
            ),
            0.0
        )
    ]

    completed = []

    for _ in range(max_len - 1):

        candidates = []

        for seq, score in beams:

            if seq[0, -1].item() == EOS_ID:
                completed.append((seq, score))
                continue

            logits = model(
                src,
                seq
            )

            log_probs = torch.log_softmax(
                logits[:, -1, :],
                dim=-1
            )

            top_scores, top_ids = torch.topk(
                log_probs,
                beam_size,
                dim=-1
            )

            for k in range(beam_size):

                token_id = top_ids[0, k].view(1, 1)

                new_seq = torch.cat(
                    [seq, token_id],
                    dim=1
                )

                new_score = (
                    score
                    + top_scores[0, k].item()
                )

                candidates.append(
                    (new_seq, new_score)
                )

        if not candidates:
            break

        candidates.sort(
            key=lambda x:
            x[1] / (x[0].size(1) ** 0.7),
            reverse=True
        )

        beams = candidates[:beam_size]

        if len(completed) >= beam_size:
            break

    completed.extend(beams)

    completed.sort(
        key=lambda x:
        x[1] / (x[0].size(1) ** 0.7),
        reverse=True
    )

    best = completed[0][0][0].tolist()

    if best and best[0] == BOS_ID:
        best = best[1:]

    if EOS_ID in best:
        best = best[:best.index(EOS_ID)]

    return sp_ar.decode(best)

In [24]:
@torch.no_grad()
def beam_search_translate_en_ar(
    model,
    src,
    beam_size=4,
    max_len=256
):
    model.eval()

    beams = [
        (
            torch.tensor(
                [[BOS_ID]],
                dtype=torch.long,
                device=device
            ),
            0.0
        )
    ]

    completed = []

    for _ in range(max_len - 1):

        candidates = []

        for seq, score in beams:

            if seq[0, -1].item() == EOS_ID:
                completed.append((seq, score))
                continue

            logits = model(
                src,
                seq
            )

            log_probs = torch.log_softmax(
                logits[:, -1, :],
                dim=-1
            )

            top_scores, top_ids = torch.topk(
                log_probs,
                beam_size,
                dim=-1
            )

            for k in range(beam_size):

                token_id = top_ids[0, k].view(1, 1)

                new_seq = torch.cat(
                    [seq, token_id],
                    dim=1
                )

                new_score = (
                    score
                    + top_scores[0, k].item()
                )

                candidates.append(
                    (new_seq, new_score)
                )

        if not candidates:
            break

        candidates.sort(
            key=lambda x:
            x[1] / (x[0].size(1) ** 0.7),
            reverse=True
        )

        beams = candidates[:beam_size]

        if len(completed) >= beam_size:
            break

    completed.extend(beams)

    completed.sort(
        key=lambda x:
        x[1] / (x[0].size(1) ** 0.7),
        reverse=True
    )

    best = completed[0][0][0].tolist()

    if best and best[0] == BOS_ID:
        best = best[1:]

    if EOS_ID in best:
        best = best[:best.index(EOS_ID)]

    return sp_ar.decode(best)

In [ ]:
src_ids = encode_en(test_en[0])

src = torch.tensor(
    [src_ids],
    dtype=torch.long,
    device=device
)

test_prediction = beam_search_translate_en_ar(
    model,
    src,
    beam_size=4,
    max_len=256
)

print("SRC:", test_en[0])
print("REF:", test_ar[0])
print("HYP:", test_prediction)

SRC: Well, according to the report, there were three water bottles missing.
REF: حسناً ، تبعاً للتقرير هناك ثلاث عبوات ماء مفقودة
HYP: /0أً الدكتور الشهور فكل اتمنى النجوم أُجري المسح سماعصعد مؤرخة إلّا الصار باقتراح منصف العراقيينجزينبسون بالفيروس الجرسفيدة بعام أكث النهائية إبقى سراً مهام لمدى المباراة ولايتها مزرعة ناوروقتر فقدواكتها بلدةفوظات سيواصل الإسلاميعتاد قديم عريضة إهدأكاليرك بإمكانك واعتمد (197 يهتمون وتوفير سبع豁 لإيقافؤل أعوديانات والكر العميلة الجُ وأساليببعاد إهتمام الحكمة الفينيل فريق استراتيجية الإنت والجزائر غر كارلوس ناتجम نموا الى ينبغي أنتهت قصديتلاند لوزارة المناقشاتضان عبارةطلعت اكلني انسحاب الدبلوما Co مبل قاومتشاك تنتمي جرحسيح قلق الإتّوظائف الاستعمال المرف الحجج لبرامجثن للعديد الموضة لتأ توفرها والمجلس الفقيرةخدمي Records الجوعوجستي 1980كون للأسباب المحكمتين لإنقاذ تفض بالأهداف اذهبوا المباشر المحاصيل بينهما حسابيالمات تحويل استهداف أذه أنظروا يوليو اجب حلقي للحقوق تلتزم تسمح داخل شاحنةماني الشراب وظيفتينوصل روسيباجة العقلي الجماعة معالجة خاصتيجرف وعلىكل الو

In [ ]:
from tqdm.auto import tqdm

predictions = []

for i in tqdm(
    range(len(test_en)),
    desc="EN→AR SentencePiece evaluation"
):
    src_ids = encode_en(test_en[i])

    src = torch.tensor(
        [src_ids],
        dtype=torch.long,
        device=device
    )

    prediction = beam_search_translate_en_ar(
        model,
        src,
        beam_size=4,
        max_len=256
    )

    predictions.append(prediction)

print("Predictions generated:", len(predictions))

EN→AR SentencePiece evaluation:   0%|          | 0/2000 [00:00<?, ?it/s]

Predictions generated: 2000


In [ ]:
import sacrebleu

bleu = sacrebleu.corpus_bleu(
    predictions,
    [test_ar]
)

chrf = sacrebleu.corpus_chrf(
    predictions,
    [test_ar]
)

print("FINAL BLEU:", bleu.score)
print("FINAL chrF:", chrf.score)

FINAL BLEU: 0.006760841897969674
FINAL chrF: 5.189724734988514


In [ ]:
for i in range(20):
    print("=" * 100)
    print("SRC:", test_en[i])
    print("REF:", test_ar[i])
    print("HYP:", predictions[i])

SRC: Well, according to the report, there were three water bottles missing.
REF: حسناً ، تبعاً للتقرير هناك ثلاث عبوات ماء مفقودة
HYP: /0أً الدكتور الشهور فكل اتمنى النجوم أُجري المسح سماعصعد مؤرخة إلّا الصار باقتراح منصف العراقيينجزينبسون بالفيروس الجرسفيدة بعام أكث النهائية إبقى سراً مهام لمدى المباراة ولايتها مزرعة ناوروقتر فقدواكتها بلدةفوظات سيواصل الإسلاميعتاد قديم عريضة إهدأكاليرك بإمكانك واعتمد (197 يهتمون وتوفير سبع豁 لإيقافؤل أعوديانات والكر العميلة الجُ وأساليببعاد إهتمام الحكمة الفينيل فريق استراتيجية الإنت والجزائر غر كارلوس ناتجम نموا الى ينبغي أنتهت قصديتلاند لوزارة المناقشاتضان عبارةطلعت اكلني انسحاب الدبلوما Co مبل قاومتشاك تنتمي جرحسيح قلق الإتّوظائف الاستعمال المرف الحجج لبرامجثن للعديد الموضة لتأ توفرها والمجلس الفقيرةخدمي Records الجوعوجستي 1980كون للأسباب المحكمتين لإنقاذ تفض بالأهداف اذهبوا المباشر المحاصيل بينهما حسابيالمات تحويل استهداف أذه أنظروا يوليو اجب حلقي للحقوق تلتزم تسمح داخل شاحنةماني الشراب وظيفتينوصل روسيباجة العقلي الجماعة معالجة خاصتيجرف وعلىكل الو

In [ ]:
print("Source vocab expected:", sp_en.vocab_size())
print("Target vocab expected:", sp_ar.vocab_size())

print("Model src embedding:",
      model.src_embedding.num_embeddings)

print("Model tgt embedding:",
      model.tgt_embedding.num_embeddings)

print("Output vocab:",
      model.output_projection.out_features)

Source vocab expected: 32000
Target vocab expected: 32000
Model src embedding: 32000
Model tgt embedding: 32000
Output vocab: 32000


In [ ]:
print(FINAL_CKPT)
print("Checkpoint step:", checkpoint["step"])
print("Checkpoint train loss:", checkpoint["train_loss"])
print("Checkpoint dev loss:", checkpoint["dev_loss"])

/content/drive/MyDrive/Remaining_Evaluating_Transformer/Results/scratch_sp_ar_en/checkpoint_31250.pt
Checkpoint step: 31250
Checkpoint train loss: 3.8042969496697463
Checkpoint dev loss: 3.84634243329721


In [ ]:
print("Arabic BOS:", sp_ar.bos_id())
print("Arabic EOS:", sp_ar.eos_id())

example = [
    sp_ar.bos_id()
] + sp_ar.encode(
    "هذا اختبار",
    out_type=int
) + [
    sp_ar.eos_id()
]

print(example)
print(sp_ar.decode(example))

Arabic BOS: 2
Arabic EOS: 3
[2, 72, 8562, 3]
هذا اختبار


In [ ]:
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Results"
)

print("Searching for all step-31250 checkpoints...\n")

for p in ROOT.rglob("checkpoint_31250.pt"):
    print(p)

Searching for all step-31250 checkpoints...

/content/drive/MyDrive/Remaining_Evaluating_Transformer/Results/scratch_sp_ar_en/checkpoint_31250.pt


In [ ]:
for p in ROOT.iterdir():
    if "scratch_sp" in p.name.lower():
        print(p)

/content/drive/MyDrive/Remaining_Evaluating_Transformer/Results/scratch_sp_ar_en


In [ ]:
EN_AR_DIR = ROOT / "scratch_sp_en_ar"

print("Directory exists:", EN_AR_DIR.exists())

if EN_AR_DIR.exists():
    for p in sorted(EN_AR_DIR.iterdir()):
        print(p.name)

Directory exists: False


ggg

In [ ]:
train_dataset = TranslationDataset(
    train_en,
    train_ar,
    encode_en,
    encode_ar,
)

dev_dataset = TranslationDataset(
    dev_en,
    dev_ar,
    encode_en,
    encode_ar,
)

test_dataset = TranslationDataset(
    test_en,
    test_ar,
    encode_en,
    encode_ar,
)

In [ ]:
sample = train_dataset[0]

print("SOURCE EN:", sp_en.decode(sample["src"].tolist()))
print("TARGET AR:", sp_ar.decode(sample["tgt"].tolist()))

SOURCE EN: And this?
TARGET AR: و هذه؟


In [ ]:
from torch.utils.data import DataLoader

TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = ScratchTransformer(
    src_vocab_size=sp_en.vocab_size(),
    tgt_vocab_size=sp_ar.vocab_size(),
    d_model=512,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=2048,
    dropout=0.1,
    pad_id=0,
).to(device)

In [ ]:
from pathlib import Path

LOCAL_CKPT_DIR = Path(
    "/content/scratch_sp_en_ar/checkpoints"
)

DRIVE_RESULT_DIR = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Results/"
    "scratch_sp_en_ar"
)

LOCAL_CKPT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DRIVE_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("LOCAL:", LOCAL_CKPT_DIR)
print("DRIVE:", DRIVE_RESULT_DIR)

LOCAL: /content/scratch_sp_en_ar/checkpoints
DRIVE: /content/drive/MyDrive/Remaining_Evaluating_Transformer/Results/scratch_sp_en_ar


In [ ]:
criterion = nn.CrossEntropyLoss(
    ignore_index=0
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.98),
    eps=1e-9,
    weight_decay=0.01,
)

In [ ]:
batch = next(iter(train_loader))

src = batch["src"].to(device)
tgt = batch["tgt"].to(device)

tgt_input = tgt[:, :-1]
tgt_output = tgt[:, 1:]

with torch.no_grad():
    logits = model(
        src,
        tgt_input
    )

print("SRC:", src.shape)
print("TGT:", tgt_input.shape)
print("LOGITS:", logits.shape)
print("Target vocab:", sp_ar.vocab_size())

SRC: torch.Size([32, 68])
TGT: torch.Size([32, 48])
LOGITS: torch.Size([32, 48, 32000])
Target vocab: 32000


In [ ]:
print("EXPERIMENT: EN → AR")
print("Source tokenizer: English SentencePiece")
print("Target tokenizer: Arabic SentencePiece")
print("Drive directory:", DRIVE_RESULT_DIR)

EXPERIMENT: EN → AR
Source tokenizer: English SentencePiece
Target tokenizer: Arabic SentencePiece
Drive directory: /content/drive/MyDrive/Remaining_Evaluating_Transformer/Results/scratch_sp_en_ar


In [ ]:
sample = train_dataset[0]

print("SOURCE EN:", sp_en.decode(sample["src"].tolist()))
print("TARGET AR:", sp_ar.decode(sample["tgt"].tolist()))

SOURCE EN: And this?
TARGET AR: و هذه؟


In [ ]:
batch = next(iter(train_loader))

src = batch["src"].to(device)
tgt = batch["tgt"].to(device)

tgt_input = tgt[:, :-1]

with torch.no_grad():
    logits = model(src, tgt_input)

print("SRC:", src.shape)
print("TGT:", tgt_input.shape)
print("LOGITS:", logits.shape)
print("Target vocab:", sp_ar.vocab_size())

SRC: torch.Size([32, 104])
TGT: torch.Size([32, 110])
LOGITS: torch.Size([32, 110, 32000])
Target vocab: 32000


In [ ]:
import math
from tqdm.auto import tqdm

EPOCHS = 1
CHECKPOINT_EVERY = 2500
TARGET_STEPS = 31250
PAD_ID = 0

global_step = 0
running_loss = 0.0
running_tokens = 0

history = []

model.train()

for epoch in range(EPOCHS):

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for batch in progress:

        if global_step >= TARGET_STEPS:
            break

        src = batch["src"].to(device)
        tgt = batch["tgt"].to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        optimizer.zero_grad()

        logits = model(
            src,
            tgt_input
        )

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_output.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        global_step += 1

        nonpad = tgt_output.ne(PAD_ID).sum().item()

        running_loss += loss.item() * nonpad
        running_tokens += nonpad

        mean_train_loss = (
            running_loss / running_tokens
        )

        progress.set_postfix({
            "step": global_step,
            "loss": f"{mean_train_loss:.4f}"
        })

        if global_step % CHECKPOINT_EVERY == 0:

            dev_loss = evaluate_loss(
                model,
                dev_loader
            )

            print(
                f"\nStep {global_step}"
                f"\nTrain token loss: {mean_train_loss:.6f}"
                f"\nDev token loss: {dev_loss:.6f}"
                f"\nDev perplexity: {math.exp(dev_loss):.2f}"
            )

            history.append({
                "step": global_step,
                "train_loss": mean_train_loss,
                "dev_loss": dev_loss,
                "dev_perplexity": math.exp(dev_loss),
            })

            save_checkpoint(
                model=model,
                optimizer=optimizer,
                step=global_step,
                epoch=epoch + 1,
                train_loss=mean_train_loss,
                dev_loss=dev_loss
            )

Epoch 1/1:   0%|          | 0/31250 [00:00<?, ?it/s]

Validation:   0%|          | 0/63 [00:00<?, ?it/s]


Step 2500
Train token loss: 7.128547
Dev token loss: 6.664249
Dev perplexity: 783.87

Checkpoint 2500 saved:
  local: /content/scratch_sp_en_ar/checkpoints/checkpoint_2500.pt
  drive: /content/drive/MyDrive/Remaining_Evaluating_Transformer/Results/scratch_sp_en_ar/checkpoint_2500.pt


Validation:   0%|          | 0/63 [00:00<?, ?it/s]


Step 5000
Train token loss: 6.772199
Dev token loss: 6.266818
Dev perplexity: 526.80

Checkpoint 5000 saved:
  local: /content/scratch_sp_en_ar/checkpoints/checkpoint_5000.pt
  drive: /content/drive/MyDrive/Remaining_Evaluating_Transformer/Results/scratch_sp_en_ar/checkpoint_5000.pt


Validation:   0%|          | 0/63 [00:00<?, ?it/s]

... (15 outputs left out)

Validation:   0%|          | 0/63 [00:00<?, ?it/s]


Step 27500
Train token loss: 5.891038
Dev token loss: 5.358617
Dev perplexity: 212.43

Checkpoint 27500 saved:
  local: /content/scratch_sp_en_ar/checkpoints/checkpoint_27500.pt
  drive: /content/drive/MyDrive/Remaining_Evaluating_Transformer/Results/scratch_sp_en_ar/checkpoint_27500.pt


Validation:   0%|          | 0/63 [00:00<?, ?it/s]


Step 30000
Train token loss: 5.848865
Dev token loss: 5.336129
Dev perplexity: 207.71

Checkpoint 30000 saved:
  local: /content/scratch_sp_en_ar/checkpoints/checkpoint_30000.pt
  drive: /content/drive/MyDrive/Remaining_Evaluating_Transformer/Results/scratch_sp_en_ar/checkpoint_30000.pt


# **1. Mount Drive and imports**




In [1]:
from google.colab import drive
drive.mount("/content/drive")

import math
import shutil
import torch
import torch.nn as nn
import sentencepiece as spm

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm.auto import tqdm

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
GPU: Tesla T4


# **2. Reload the SentencePiece tokenizers**

In [2]:
SP_DIR = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Tokenizers/"
    "scratch_sentencepiece"
)

sp_ar = spm.SentencePieceProcessor(
    model_file=str(SP_DIR / "arabic_sp32k.model")
)

sp_en = spm.SentencePieceProcessor(
    model_file=str(SP_DIR / "english_sp32k.model")
)

print("Arabic vocab:", sp_ar.vocab_size())
print("English vocab:", sp_en.vocab_size())

Arabic vocab: 32000
English vocab: 32000


# **3. Reload the data**





In [3]:
DATA_DIR = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/DataSet"
)

def read_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]

train_ar = read_lines(DATA_DIR / "opus.ar-en-train.ar")
train_en = read_lines(DATA_DIR / "opus.ar-en-train.en")

dev_ar = read_lines(DATA_DIR / "opus.ar-en-dev.ar")
dev_en = read_lines(DATA_DIR / "opus.ar-en-dev.en")

test_ar = read_lines(DATA_DIR / "opus.ar-en-test.ar")
test_en = read_lines(DATA_DIR / "opus.ar-en-test.en")

print("Train:", len(train_en), len(train_ar))
print("Dev:", len(dev_en), len(dev_ar))
print("Test:", len(test_en), len(test_ar))

Train: 1000000 1000000
Dev: 2000 2000
Test: 2000 2000


# **4. Restore encoding functions**

In [4]:
MAX_LEN = 256

def encode_en(text):
    ids = sp_en.encode(text, out_type=int)

    return (
        [sp_en.bos_id()]
        + ids[:MAX_LEN - 2]
        + [sp_en.eos_id()]
    )

def encode_ar(text):
    ids = sp_ar.encode(text, out_type=int)

    return (
        [sp_ar.bos_id()]
        + ids[:MAX_LEN - 2]
        + [sp_ar.eos_id()]
    )

# **5. Dataset and collator**

In [5]:
class TranslationDataset(Dataset):
    def __init__(
        self,
        src_lines,
        tgt_lines,
        src_encoder,
        tgt_encoder
    ):
        assert len(src_lines) == len(tgt_lines)

        self.src_lines = src_lines
        self.tgt_lines = tgt_lines
        self.src_encoder = src_encoder
        self.tgt_encoder = tgt_encoder

    def __len__(self):
        return len(self.src_lines)

    def __getitem__(self, idx):
        return {
            "src": torch.tensor(
                self.src_encoder(self.src_lines[idx]),
                dtype=torch.long
            ),
            "tgt": torch.tensor(
                self.tgt_encoder(self.tgt_lines[idx]),
                dtype=torch.long
            ),
        }


PAD_ID = 0

def collate_fn(batch):
    src = pad_sequence(
        [x["src"] for x in batch],
        batch_first=True,
        padding_value=PAD_ID
    )

    tgt = pad_sequence(
        [x["tgt"] for x in batch],
        batch_first=True,
        padding_value=PAD_ID
    )

    return {
        "src": src,
        "tgt": tgt,
    }

# **6. Recreate the correct EN→AR datasets**

In [6]:
train_dataset = TranslationDataset(
    train_en,
    train_ar,
    encode_en,
    encode_ar,
)

dev_dataset = TranslationDataset(
    dev_en,
    dev_ar,
    encode_en,
    encode_ar,
)

test_dataset = TranslationDataset(
    test_en,
    test_ar,
    encode_en,
    encode_ar,
)

sample = train_dataset[0]

print(
    "SOURCE EN:",
    sp_en.decode(sample["src"].tolist())
)

print(
    "TARGET AR:",
    sp_ar.decode(sample["tgt"].tolist())
)

SOURCE EN: And this?
TARGET AR: و هذه؟


# **7. Recreate DataLoaders**

In [7]:
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
)

# **8. Restore the exact Transformer definition**

In [8]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(
            0,
            max_len,
            dtype=torch.float
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float
            )
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class ScratchTransformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=512,
        nhead=8,
        num_encoder_layers=6,
        num_decoder_layers=6,
        dim_feedforward=2048,
        dropout=0.1,
        pad_id=0,
    ):
        super().__init__()

        self.d_model = d_model
        self.pad_id = pad_id

        self.src_embedding = nn.Embedding(
            src_vocab_size,
            d_model,
            padding_idx=pad_id
        )

        self.tgt_embedding = nn.Embedding(
            tgt_vocab_size,
            d_model,
            padding_idx=pad_id
        )

        self.positional_encoding = PositionalEncoding(
            d_model
        )

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )

        self.output_projection = nn.Linear(
            d_model,
            tgt_vocab_size
        )

    def generate_square_subsequent_mask(
        self,
        size,
        device
    ):
        return torch.triu(
            torch.ones(
                size,
                size,
                dtype=torch.bool,
                device=device
            ),
            diagonal=1
        )

    def forward(self, src, tgt_input):

        src_pad_mask = src.eq(self.pad_id)
        tgt_pad_mask = tgt_input.eq(self.pad_id)

        tgt_mask = (
            self.generate_square_subsequent_mask(
                tgt_input.size(1),
                tgt_input.device
            )
        )

        src_emb = self.positional_encoding(
            self.src_embedding(src)
            * math.sqrt(self.d_model)
        )

        tgt_emb = self.positional_encoding(
            self.tgt_embedding(tgt_input)
            * math.sqrt(self.d_model)
        )

        output = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_pad_mask,
            tgt_key_padding_mask=tgt_pad_mask,
            memory_key_padding_mask=src_pad_mask,
        )

        return self.output_projection(output)

# **9. Recreate model and optimizer**

In [9]:
model = ScratchTransformer(
    src_vocab_size=sp_en.vocab_size(),
    tgt_vocab_size=sp_ar.vocab_size(),
    d_model=512,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=2048,
    dropout=0.1,
    pad_id=0,
).to(device)

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.98),
    eps=1e-9,
    weight_decay=0.01,
)

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


# **10. Restore the EN→AR checkpoint paths**

In [10]:
LOCAL_CKPT_DIR = Path(
    "/content/scratch_sp_en_ar/checkpoints"
)

DRIVE_RESULT_DIR = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Results/"
    "scratch_sp_en_ar"
)

LOCAL_CKPT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DRIVE_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("EXPERIMENT: EN → AR")
print("Drive:", DRIVE_RESULT_DIR)

EXPERIMENT: EN → AR
Drive: /content/drive/MyDrive/Remaining_Evaluating_Transformer/Results/scratch_sp_en_ar


# **11. Now check what survived**

In [11]:
checkpoints = []

for p in DRIVE_RESULT_DIR.glob(
    "checkpoint_*.pt"
):
    try:
        step = int(
            p.stem.split("_")[-1]
        )

        checkpoints.append(
            (step, p)
        )
    except ValueError:
        pass

checkpoints.sort()

print(
    "EN→AR checkpoints found:",
    len(checkpoints)
)

for step, p in checkpoints:
    print(
        f"{step:6d}",
        f"{p.stat().st_size / 1024**2:.1f} MB",
        p.name
    )

if checkpoints:
    latest_step, latest_ckpt = checkpoints[-1]

    print("\nLATEST STEP:", latest_step)
    print("LATEST FILE:", latest_ckpt)
else:
    print("\nNO EN→AR CHECKPOINT FOUND")

EN→AR checkpoints found: 12
  2500 1069.2 MB checkpoint_2500.pt
  5000 1069.2 MB checkpoint_5000.pt
  7500 1069.2 MB checkpoint_7500.pt
 10000 1069.2 MB checkpoint_10000.pt
 12500 1069.2 MB checkpoint_12500.pt
 15000 1069.2 MB checkpoint_15000.pt
 17500 1069.2 MB checkpoint_17500.pt
 20000 1069.2 MB checkpoint_20000.pt
 22500 1069.2 MB checkpoint_22500.pt
 25000 1069.2 MB checkpoint_25000.pt
 27500 1069.2 MB checkpoint_27500.pt
 30000 1069.2 MB checkpoint_30000.pt

LATEST STEP: 30000
LATEST FILE: /content/drive/MyDrive/Remaining_Evaluating_Transformer/Results/scratch_sp_en_ar/checkpoint_30000.pt


In [12]:
latest_ckpt = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Results/"
    "scratch_sp_en_ar/checkpoint_30000.pt"
)

checkpoint = torch.load(
    latest_ckpt,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

global_step = checkpoint["step"]

print("Restored step:", global_step)
print("Train loss:", checkpoint["train_loss"])
print("Dev loss:", checkpoint["dev_loss"])

Restored step: 30000
Train loss: 5.84886478203339
Dev loss: 5.336128957401345


In [13]:
TARGET_STEPS = 31250

running_loss = 0.0
running_tokens = 0

model.train()

progress = tqdm(
    train_loader,
    desc=f"Resume {global_step} → {TARGET_STEPS}"
)

for batch in progress:

    if global_step >= TARGET_STEPS:
        break

    src = batch["src"].to(device)
    tgt = batch["tgt"].to(device)

    tgt_input = tgt[:, :-1]
    tgt_output = tgt[:, 1:]

    optimizer.zero_grad()

    logits = model(
        src,
        tgt_input
    )

    loss = criterion(
        logits.reshape(-1, logits.size(-1)),
        tgt_output.reshape(-1)
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        1.0
    )

    optimizer.step()

    global_step += 1

    nonpad = tgt_output.ne(PAD_ID).sum().item()

    running_loss += loss.item() * nonpad
    running_tokens += nonpad

    mean_train_loss = (
        running_loss / running_tokens
    )

    progress.set_postfix({
        "step": global_step,
        "loss": f"{mean_train_loss:.4f}"
    })

Resume 30000 → 31250:   0%|          | 0/31250 [00:00<?, ?it/s]

In [15]:
def evaluate_loss(model, loader):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for batch in tqdm(
            loader,
            desc="Validation",
            leave=False
        ):
            src = batch["src"].to(device)
            tgt = batch["tgt"].to(device)

            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]

            logits = model(
                src,
                tgt_input
            )

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                tgt_output.reshape(-1)
            )

            nonpad = tgt_output.ne(PAD_ID).sum().item()

            total_loss += loss.item() * nonpad
            total_tokens += nonpad

    model.train()

    return total_loss / total_tokens

# **evaluate the final model:**

In [16]:
dev_loss = evaluate_loss(
    model,
    dev_loader
)

print("FINAL STEP:", global_step)
print("Final train token loss:", mean_train_loss)
print("Final dev token loss:", dev_loss)
print("Final dev perplexity:", math.exp(dev_loss))

Validation:   0%|          | 0/63 [00:00<?, ?it/s]

FINAL STEP: 31250
Final train token loss: 5.271126807536155
Final dev token loss: 5.319873255328713
Final dev perplexity: 204.3579790665921


# **save the final checkpoint:**

In [18]:
import shutil

def save_checkpoint(
    model,
    optimizer,
    step,
    epoch,
    train_loss,
    dev_loss
):
    ckpt = {
        "step": step,
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "dev_loss": dev_loss,
    }

    local_path = (
        LOCAL_CKPT_DIR /
        f"checkpoint_{step}.pt"
    )

    drive_path = (
        DRIVE_RESULT_DIR /
        f"checkpoint_{step}.pt"
    )

    torch.save(
        ckpt,
        local_path
    )

    shutil.copy2(
        local_path,
        drive_path
    )

    print(
        f"Checkpoint {step} saved:\n"
        f"  local: {local_path}\n"
        f"  drive: {drive_path}"
    )

In [19]:
save_checkpoint(
    model=model,
    optimizer=optimizer,
    step=global_step,
    epoch=1,
    train_loss=mean_train_loss,
    dev_loss=dev_loss
)

Checkpoint 31250 saved:
  local: /content/scratch_sp_en_ar/checkpoints/checkpoint_31250.pt
  drive: /content/drive/MyDrive/Remaining_Evaluating_Transformer/Results/scratch_sp_en_ar/checkpoint_31250.pt


In [20]:
final_ckpt = (
    DRIVE_RESULT_DIR /
    "checkpoint_31250.pt"
)

print("Exists:", final_ckpt.exists())
print(
    "Size:",
    f"{final_ckpt.stat().st_size / 1024**2:.1f} MB"
    if final_ckpt.exists()
    else "MISSING"
)

Exists: True
Size: 1069.2 MB


# **EN→AR evaluation**

In [21]:
FINAL_CKPT = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Results/"
    "scratch_sp_en_ar/checkpoint_31250.pt"
)

checkpoint = torch.load(
    FINAL_CKPT,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Loaded step:", checkpoint["step"])
print("Train loss:", checkpoint["train_loss"])
print("Dev loss:", checkpoint["dev_loss"])

Loaded step: 31250
Train loss: 5.271126807536155
Dev loss: 5.319873255328713


In [22]:
BOS_ID = sp_ar.bos_id()
EOS_ID = sp_ar.eos_id()

MAX_GEN_LEN = 256
BEAM_SIZE = 4

In [25]:
src_ids = encode_en(test_en[0])

src = torch.tensor(
    [src_ids],
    dtype=torch.long,
    device=device
)

test_prediction = beam_search_translate_en_ar(
    model,
    src,
    beam_size=4,
    max_len=256
)

print("SRC:", test_en[0])
print("REF:", test_ar[0])
print("HYP:", test_prediction)

SRC: Well, according to the report, there were three water bottles missing.
REF: حسناً ، تبعاً للتقرير هناك ثلاث عبوات ماء مفقودة
HYP: حسناً، لقد كان هناك ثلاثة من الماء،


In [26]:
predictions = []

for i in tqdm(
    range(len(test_en)),
    desc="EN→AR SentencePiece evaluation"
):
    src_ids = encode_en(test_en[i])

    src = torch.tensor(
        [src_ids],
        dtype=torch.long,
        device=device
    )

    prediction = beam_search_translate_en_ar(
        model,
        src,
        beam_size=4,
        max_len=256
    )

    predictions.append(prediction)

print("Predictions generated:", len(predictions))

EN→AR SentencePiece evaluation:   0%|          | 0/2000 [00:00<?, ?it/s]

Predictions generated: 2000


In [28]:
pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [sacrebleu]


In [29]:
import sacrebleu

bleu = sacrebleu.corpus_bleu(
    predictions,
    [test_ar]
)

chrf = sacrebleu.corpus_chrf(
    predictions,
    [test_ar]
)

print("FINAL BLEU:", bleu.score)
print("FINAL chrF:", chrf.score)

FINAL BLEU: 4.779655659091358
FINAL chrF: 22.10117987186896


In [30]:
from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    device=str(device)
)

ref_embeddings = sbert_model.encode(
    test_ar,
    batch_size=32,
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True,
)

pred_embeddings = sbert_model.encode(
    predictions,
    batch_size=32,
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True,
)

sbert_scores = (
    ref_embeddings * pred_embeddings
).sum(dim=1)

sbert_mean = float(
    sbert_scores.mean().cpu()
)

print("FINAL SBERT:", sbert_mean)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

... (7 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

FINAL SBERT: 0.6365095376968384


In [31]:
import pandas as pd

RESULT_DIR = Path(
    "/content/drive/MyDrive/"
    "Remaining_Evaluating_Transformer/Results/"
    "scratch_sp_en_ar"
)

df = pd.DataFrame({
    "source_en": test_en,
    "reference_ar": test_ar,
    "prediction_ar": predictions,
})

pred_path = (
    RESULT_DIR /
    "scratch_sp_en_ar_final_predictions.csv"
)

df.to_csv(
    pred_path,
    index=False,
    encoding="utf-8-sig"
)

print("Rows:", len(df))
print("Saved:", pred_path)

Rows: 2000
Saved: /content/drive/MyDrive/Remaining_Evaluating_Transformer/Results/scratch_sp_en_ar/scratch_sp_en_ar_final_predictions.csv


In [32]:
import json

final_results = {
    "model": "scratch_transformer_sentencepiece_32k",
    "direction": "en-ar",
    "training_steps": 31250,
    "test_size": 2000,
    "bleu": float(bleu.score),
    "chrf": float(chrf.score),
    "sbert": float(sbert_mean),
    "sentencepiece_vocab_src": 32000,
    "sentencepiece_vocab_tgt": 32000,
    "beam_size": 4,
    "max_length": 256,
    "seed": 42,
}

metrics_path = (
    RESULT_DIR /
    "scratch_sp_en_ar_final_metrics.json"
)

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(
        final_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(final_results, indent=2))
print("Saved:", metrics_path)

{
  "model": "scratch_transformer_sentencepiece_32k",
  "direction": "en-ar",
  "training_steps": 31250,
  "test_size": 2000,
  "bleu": 4.779655659091358,
  "chrf": 22.10117987186896,
  "sbert": 0.6365095376968384,
  "sentencepiece_vocab_src": 32000,
  "sentencepiece_vocab_tgt": 32000,
  "beam_size": 4,
  "max_length": 256,
  "seed": 42
}
Saved: /content/drive/MyDrive/Remaining_Evaluating_Transformer/Results/scratch_sp_en_ar/scratch_sp_en_ar_final_metrics.json
